# siRNA Patent Landscape Pipeline
### From EPO patent records to a structured table of siRNA activity data

**What this notebook does.** It finds patents about small interfering RNA (siRNA) at the European Patent Office (EPO), keeps the relevant ones, downloads their full text, and converts the experimental tables inside those patents into clean CSV files that are ready for analysis.

**Why it exists.** Patents hold a large amount of siRNA activity data (duplex sequences, cell lines, doses, percent knockdown, IC50 values) that never reaches public databases. The data sits inside long free-text documents and irregularly formatted tables. This pipeline automates the path from a patent number to one row per measurement.

**What you need.** An EPO OPS account (free tier) and at least one Groq API key (free tier). Everything else is installed in Section 0.

### How to run it

1. Run the cells from top to bottom the first time. Each section reads the files written by the section before it.
2. Sections are also independent. If the input files are already in the working directory, any section can be run on its own.
3. All API keys are typed once, in the **Credentials** cell. No key is written into the notebook file.
4. Sections 4 to 7 are slow by design. EPO enforces an 8 second pause between requests, and the Groq free tier limits how fast the LLM calls can run.

### Pipeline at a glance

| Section | Stage | Script and entry point | Reads | Writes |
|---|---|---|---|---|
| 1 | Patent ID extraction | `epo_api_codes.download_patent_ids`, `epo_api_terms.download_patent_ids` | EPO OPS (live) | `EPO_siRNA_IDs_<years>_codes_only.csv`, `..._terms_only.csv`, `..._only_applicant_Alnylam.csv` |
| 2 | Bibliographic metadata | `epo_api_Metadata.fetch_biblio_from_csv` | one ID CSV from Section 1 | `<input name>_metadata.csv` |
| 3 | Tier filtering | `epo_filter.apply_filters` | the metadata CSV | `..._metadata_filtered.csv` |
| 4 | Full-text XML download | `xml_download.download_eps_xmls_with_ops` | one ID CSV | `eps_xmls/*.xml`, `successful_downloads.csv`, `not_in_eps.csv` |
| 5 | Table isolation | `table.extract_tables_from_patent` | `eps_xmls/*.xml` | `isolated_tables/*.xml` |
| 6 | XML to CSV, header cleaning (LLM) | `xml_to_csv.convert_directory` | `isolated_tables/` | `csv_output/*_tables.csv`, `csv_output/*_context.txt` |
| 7 | Primary table assembly (LLM) | `xml_to_primary_table.build_primary_table` | `csv_output/` | `primary_table*.csv`, `primary_ic50_table*.csv`, `primary_cell_viability_table*.csv` |

Sections 1 to 4 use the **EPO OPS** API. Sections 6 and 7 use the **Groq** LLM API.

### Scripts used by this notebook

Imported directly, one per stage:

`epo_api_codes.py`, `epo_api_terms.py`, `epo_api_Metadata.py`, `epo_filter.py`, `xml_download.py`, `table.py`, `xml_to_csv.py`, `xml_to_primary_table.py`

Imported indirectly, loaded by `xml_to_primary_table.py`, so they must sit in the same folder:

`core.py` (Groq client and shared helpers), `classification.py` (routing of each table to a target schema), `sql.py` (prompt templates and DuckDB query building)

All scripts must be in the same folder as this notebook.

## 0. Environment setup

Run this once per environment. These are the only external packages the pipeline needs.

| Package | Used for |
|---|---|
| `requests` | HTTP calls to the EPO OPS and European Publication Server (EPS) APIs |
| `pandas` | all tabular data handling |
| `beautifulsoup4` and `lxml` | XML parsing during table isolation |
| `groq` | client for the Groq LLM API (header cleaning and table assembly) |
| `duckdb` | local SQL engine that runs the LLM-generated `SELECT` queries |
| `openpyxl` | optional, lets pandas read and write Excel files for manual inspection |

If the install upgrades a package that is already loaded, restart the kernel before continuing.

In [2]:
%pip install requests pandas beautifulsoup4 lxml groq duckdb openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Credentials (run this first)

Every API key used in this notebook is set here, once, and reused by all later sections.

The cell reads environment variables first. If a variable is not set, it asks for the value with `getpass`, so the key is typed at run time and never saved inside the notebook file. That keeps the notebook safe to share or commit.

To skip the prompts, set the variables before starting Jupyter:

```bash
export EPO_CONSUMER_KEY=your_key
export EPO_CONSUMER_SECRET=your_secret
export GROQ_API_KEYS=key1,key2,key3
```

Two notes:

- EPO OPS keys come from the OPS developer portal. The free tier allows 4 GB of downloaded data per week.
- `GROQ_API_KEYS` accepts one key or several keys separated by commas, and the scripts rotate between them. Groq free-tier limits apply per account, so extra keys only raise throughput if they belong to different accounts.

In [2]:
import os
from getpass import getpass

def _credential(env_name, prompt):
    """Return a credential from the environment, or prompt for it (never stored)."""
    return os.getenv(env_name) or getpass(prompt)

# --- EPO OPS (Sections 1-5) ---
CONSUMER_KEY    = _credential("EPO_CONSUMER_KEY",    "EPO OPS consumer key: ")
CONSUMER_SECRET = _credential("EPO_CONSUMER_SECRET", "EPO OPS consumer secret: ")

# --- Groq (Sections 7-8). One or more keys, comma-separated. ---
# Groq free-tier limits are per ACCOUNT, so extra keys only raise throughput
# if they come from different Groq accounts.
GROQ_API_KEYS = _credential("GROQ_API_KEYS", "Groq API key(s), comma-separated: ")

print("Credentials loaded (EPO OPS + Groq).")

Credentials loaded (EPO OPS + Groq).


## 1. Patent identifier extraction

**Reads:** EPO OPS (live).  **Writes:** `EPO_siRNA_IDs_<start>_<end>_codes_only.csv`, `..._terms_only.csv`, `..._only_applicant_Alnylam.csv`.

This first stage collects patent **family identifiers** only. No metadata, abstracts or full texts are requested yet, so the download stays small. Each output row holds `Patent_ID`, `Family_ID` and `Country`.

Three searches run one after the other, and each is independent of the others:

- **1a. Classification codes** (`epo_api_codes.py`): what examiners tagged as RNA interference.
- **1b. Title and abstract keywords** (`epo_api_terms.py`): what the applicants actually wrote.
- **1c. Applicant name** (`epo_api_codes.py` with `only_applicant=True`): the full Alnylam portfolio, used as a reference set.

> **Output filenames encode the years.** `download_patent_ids` builds the output name from `start_year` and `end_year`, so changing the years changes the filename that every later section has to read. As written, the code and Alnylam searches use 2022 to 2025, while the keyword search uses 2001 to 2026 and therefore writes `EPO_siRNA_IDs_2001_2026_terms_only.csv`.

### 1a. Strategy A, CPC/IPC classification codes

This search queries the OPS search service by Cooperative and International Patent Classification code. These codes are assigned by examiners, so they are a structured and fairly reliable signal of technical content. The main anchor for siRNA is `C12N 15/113` (RNA interference and small interfering RNA). Additional codes cover backbone modification chemistry, lipid nanoparticle delivery, and disease-specific applications.

Each code is sent as its own CQL query, which keeps every response under the OPS limit of 2000 results. A query that still exceeds the limit is split automatically into monthly windows, and then into daily windows. Results are deduplicated at family level using a country priority table (EP first, then other national offices, then WO and US), so each family contributes exactly one row.

The call below uses `CONSUMER_KEY` and `CONSUMER_SECRET` from the Credentials cell. The optional quota check prints how much of the weekly 4 GB free-tier allowance has been used so far.

In [ ]:
from sirna_pipeline.epo import search_codes as epo_api_codes

# Optional quota check - current weekly data use vs the 4 GB free-tier limit
print("Checking EPO OPS quota...")
epo_api_codes.check_epo_quota(CONSUMER_KEY, CONSUMER_SECRET)

# CPC/IPC-based extraction across all applicants, 2022-2025
print("\nStarting CPC/IPC extraction...")
df_codes = epo_api_codes.download_patent_ids(
    consumer_key=CONSUMER_KEY,
    consumer_secret=CONSUMER_SECRET,
    start_year=2022,
    end_year=2025,
    applicant_filter=None,   # no applicant restriction - full siRNA landscape
    only_applicant=False,    # query driven by CPC/IPC codes
)
print(df_codes.head())

Checking EPO OPS quota...


NameError: name 'CONSUMER_KEY' is not defined

### 1b. Strategy B, title and abstract keywords

The second search matches free-text terms in the title and abstract fields: `siRNA`, `small interfering RNA`, `RNA interference`, `RNAi`, plus chemically specific backbone-modification variants.

It complements Strategy A in two ways. It catches patents that are clearly about siRNA in their wording but have not yet received the `C12N 15/113` code, which is common for recent applications that have not been examined yet. It also catches patents filed under a broader parent code. Because it ignores examiner classification entirely, it provides an independent recall signal.

Query slicing, rate limiting and family deduplication work exactly as in Strategy A. The script is `epo_api_terms.py`.

In [ ]:
from sirna_pipeline.epo import search_terms as epo_api_terms

print("Checking EPO OPS quota...")
epo_api_terms.check_epo_quota(CONSUMER_KEY, CONSUMER_SECRET)

print("\nStarting keyword (title/abstract) extraction...")
df_terms = epo_api_terms.download_patent_ids(
    consumer_key=CONSUMER_KEY,
    consumer_secret=CONSUMER_SECRET,
    start_year=2001,
    end_year=2026,
    applicant_filter=None,   # no applicant restriction
)
print(df_terms.head())

Checking EPO OPS quota...
      EPO OPS API - STATUS DASHBOARD

[DATA CONSUMPTION]
Weekly Volume Used: 2.8576% (114.30 MB / 4000 MB)
Quota Remaining:    97.1424%
Current API Load:   15 requests/minute

---------------------------------------------
[OK] Server clear. Safe to proceed with extraction.
---------------------------------------------

Starting keyword (title/abstract) extraction...

=== STARTING EPO ID EXTRACTION (2001 - 2026) ===
[INFO] 27 independent query conditions.
[INFO] Exclusions active: C12N15/115 (Aptamers), C12N15/117 (Immunomodulatory)

[INFO] Processing year 2001...
  -> Query 1/27: (ta=siRNA* NOT (cpc="C12N15/115" OR cpc="C12N15/117"))...

[AUTH] Generating a new EPO access token...
[WARNING 404] Slice not found (attempt 1/4). Retrying...
[WARNING 404] Slice not found (attempt 2/4). Retrying...
[WARNING 404] Slice not found (attempt 3/4). Retrying...
  -> Query 2/27: (ta=RNAi* NOT (cpc="C12N15/115" OR cpc="C12N15/117"))...
[INFO] Year 2001: 8 results. Fetching y

### 1c. Reference corpus, the full Alnylam Pharmaceuticals portfolio

As an external validation set, the complete Alnylam portfolio is retrieved by querying the applicant-name field directly (`applicant_filter="Alnylam*"`, `only_applicant=True`). This bypasses all classification and keyword constraints, and the wildcard matches the different Alnylam entity names used across filings.

Alnylam is the leading siRNA company and the originator of the first approved siRNA therapeutics (patisiran, givosiran, lumasiran, inclisiran, vutrisiran), so nearly every family in its portfolio is siRNA relevant. That makes the portfolio a high-confidence benchmark for recall: any Alnylam family missing from the output of Strategy A or B is a relevant family that strategy failed to find.

In [4]:
# Alnylam full portfolio - applicant-name query, no CPC/IPC filter
print("Extracting Alnylam full portfolio...")
df_alnylam = epo_api_codes.download_patent_ids(
    consumer_key=CONSUMER_KEY,
    consumer_secret=CONSUMER_SECRET,
    start_year=2022,
    end_year=2025,
    applicant_filter="Alnylam*",   # wildcard matches all Alnylam entity names
    only_applicant=True,            # name-driven query only - ignores CPC codes
)
print(df_alnylam.head())

Extracting Alnylam full portfolio...

=== STARTING EPO ID EXTRACTION (2022 - 2025) ===
[INFO] 1 independent query conditions.
[INFO] Applicant filter: Alnylam* (Only Applicant: True)

[INFO] Processing year 2022...
  -> Query 1/1: ...

[AUTH] Generating a new EPO access token...
[INFO] Year 2022: 153 results. Fetching yearly...
[INFO] Requested: 1-100 | Received: 1-100 | Total expected: 153
[INFO] Requested: 101-153 | Received: 101-153 | Total expected: 153
[INFO] Raw records accumulated (including cross-query duplicates): 153
[INFO] After family deduplication: 153 unique IDs
[INFO] Records filtered (cross-query duplicates + prior-year skips): 0
[INFO] Families tracked globally so far: 153
  [AUTOSAVE] Year 2022: 153 IDs saved to EPO_IDs_AutoSave_2022.csv

[INFO] Processing year 2023...
  -> Query 1/1: ...
[INFO] Year 2023: 181 results. Fetching yearly...
[INFO] Requested: 1-100 | Received: 1-100 | Total expected: 181
[INFO] Requested: 101-181 | Received: 101-181 | Total expected: 181


## 2. Bibliographic metadata enrichment

**Reads:** one ID CSV from Section 1.  **Writes:** the same filename with `_metadata` appended, for example `EPO_siRNA_IDs_2022_2025_terms_only_metadata.csv`.

Section 1 produced lean ID lists. This stage adds the full bibliographic record for each family through the OPS `/biblio` endpoint.

Fields retrieved: earliest priority date, publication date, applicant names, invention title (English preferred, with a fallback), abstract (English preferred), IPC codes and CPC codes.

How the stage protects a long run:

- IDs are sent in batches of 100, which is the `/biblio` hard limit.
- A failing batch is retried up to three times, and then each ID in it is retried on its own, so one bad record cannot cost the other 99.
- If a record comes back with no abstract, a second call to `/abstract` is made for that ID alone.
- An adaptive rate limiter lengthens the pause between batches when the server signals congestion, and triggers a session cooldown after repeated throttling.
- A final pass deduplicates by `Family_ID`, preferring records with a usable English abstract and a Latin-script title, then the oldest priority date.

> **Choosing the input.** Enriching the full landscape extracts is expensive. They are much larger and contain many families that turn out to be irrelevant, so enriching all of them before the filters are tuned would spend the weekly 4 GB quota for little gain. The Alnylam CSV is the cheaper, pre-validated alternative.
>
> The cell below is currently set to `EPO_siRNA_IDs_2022_2025_terms_only.csv`, the keyword extract. Change `ids_csv` to use a different corpus, and check that the filename matches the years actually used in Section 1.

In [ ]:
from sirna_pipeline.epo.biblio import fetch_biblio_from_csv

df_metadata = fetch_biblio_from_csv(
    ids_csv         = "EPO_siRNA_IDs_2022_2025_terms_only.csv",
    consumer_key    = CONSUMER_KEY,
    consumer_secret = CONSUMER_SECRET,
)
df_metadata.head()


=== STARTING EPO METADATA FETCH ===
[INFO] Input file : EPO_siRNA_IDs_2022_2025_terms_only.csv
[INFO] Patent IDs : 7405
[INFO] Batches : 75 x 100 IDs per batch

[AUTH] Generating a new EPO access token...
[INFO] Batch 1 complete — 100 records fetched so far.
[INFO] Batch 2 complete — 200 records fetched so far.
  [WARNING] Abstract fallback failed for EP.4381069.A1: ReadTimeout: HTTPSConnectionPool(host='ops.epo.org', port=443): Read timed out. (read timeout=15)
[INFO] Batch 3 complete — 300 records fetched so far.

[AUTH] Generating a new EPO access token...
[INFO] Batch 4 complete — 400 records fetched so far.
[INFO] Batch 5 complete — 500 records fetched so far.
[INFO] Batch 6 complete — 600 records fetched so far.
[INFO] Batch 7 complete — 700 records fetched so far.
[INFO] Batch 8 complete — 800 records fetched so far.
[INFO] Batch 9 complete — 900 records fetched so far.
[INFO] Batch 10 complete — 1000 records fetched so far.
[INFO] Batch 11 complete — 1100 records fetched so fa

,Patent_ID,Country,Number,Kind,Family_ID,Priority_Date,Publication_Date,Applicant,Title,Abstract,IPCs,CPCs
1550,US2022275366A1,US,2022275366,A1,83006944,20010518,20220901,SIRNA THERAPEUTICS INC [US] | SIRNA THERAPEUTI...,RNA INTERFERENCE MEDIATED INHIBITION OF GENE E...,The present invention concerns methods and rea...,,"A61K38/00, A61K47/54, A61K47/544, A61K47/549, ..."
420,US2022056441A1,US,2022056441,A1,53183180,20020220,20220224,SIRNA THERAPEUTICS INC [US] | SIRNA THERAPEUTI...,RNA INTERFERENCE MEDIATED INHIBITION OF GENE E...,The present invention concerns methods and rea...,,"C07H21/02, C12N15/111, C12N15/113, C12N15/1131..."
431,US2022112494A1,US,2022112494,A1,32046073,20020925,20220414,UNIV MASSACHUSETTS [US] | UNIVERSITY OF MASSAC...,IN VIVO GENE SILENCING BY CHEMICALLY MODIFIED ...,The present invention provides compositions fo...,,"A01K2217/075, A61K38/00, A61K48/00, C07D213/69..."
437,US2022315922A1,US,2022315922,A1,50773796,20021114,20221006,THERMO FISHER SCIENTIFIC INC [US] | Thermo Fis...,Methods and Compositions for Selecting siRNA o...,Efficient sequence specific gene silencing is ...,,"A61K31/713, A61K48/00, C12N15/1048, C12N15/111..."
682,US2022062286A1,US,2022062286,A1,27772701,20030725,20220303,UNIV SHEFFIELD [GB] | The University of Sheffield,USE OF RNAI INHIBITING PARP ACTIVITY FOR THE M...,The present invention relates to the use of an...,,"A61K31/472, A61K31/517, A61K31/5517, A61K31/70..."


## 3. Classification into tiers

**Reads:** a metadata CSV from Section 2.  **Writes:** `..._metadata_filtered.csv`.

`epo_filter.apply_filters` sorts every patent into one of eight tiers (1, 2, 3, 4A, 4B, 5, 6, 7) from two signals: siRNA wording in the title and abstract, and the structural `C12N15/113` classification anchor. Tiers are tested in priority order and each patent receives the highest tier it qualifies for, so the tiers never overlap.

Nothing is deleted. Every patent is written out with its tier and a direct Espacenet link, which keeps the dataset complete and makes manual curation transparent. The tiers guide the review, they do not replace it. In the output CSV each tier block is preceded by a blank separator row carrying the tier name, which makes the file easy to read by eye. Filter on the tier column before any further processing.

| Tier | Criterion | Recommended action |
|---|---|---|
| **Tier 1** | siRNA wording **and** the `C12N15/113` anchor | Core dataset, include without further review |
| **Tier 2** | Wording only, anchor absent (not yet classified, or filed under a broader code) | High confidence, include. A missing CPC code is not disqualifying |
| **Tier 3** | Anchor only, no wording (abstract missing, non-English or uninformative) | Check on Espacenet before including |
| **Tier 4A** | siRNA wording or anchor, **plus** a competing-technology term (aptamer, antisense oligonucleotide, CRISPR) | Mixed technology, review to identify the primary one |
| **Tier 4B** | Diagnostic or biomarker language, no therapeutic application | Likely out of scope for a therapeutics analysis |
| **Tier 5** | Agricultural, veterinary or pest-control application **with** the anchor | Depends on scope (RNAi in plants and insects) |
| **Tier 6** | No wording and no anchor | Likely irrelevant, lowest priority |
| **Tier 7** | Agricultural or veterinary application **without** the anchor | Likely irrelevant |


**Known limitation.** Keyword and CPC rules cannot handle negation or ambiguous phrasing. For now the rule-based system gives enough signal to move on to full-text processing at a defensible confidence level.

In [ ]:
from sirna_pipeline.filtering.tiers import apply_filters

input_csv  = "EPO_siRNA_IDs_2022_2025_terms_only_metadata.csv"
output_csv = "EPO_siRNA_IDs_2022_2025_terms_only_metadata_filtered.csv"

result_df = apply_filters(
    raw_data=input_csv,
    output_filename=output_csv,
    csv_sep=";",
    csv_encoding="utf-8-sig",
)
display(result_df.head(20))

[INFO] Reading CSV from disk: EPO_siRNA_IDs_2022_2025_terms_only_metadata.csv

=== STARTING PATENT CLASSIFICATION ===
[INFO] No patents will be deleted — all records go to the output CSV.

[SUCCESS] Classification complete.
  Total input patents : 7405

  TIER 1 — siRNA Confirmed (Text + CPC)             1588  ███████████████
  TIER 2 — siRNA Confirmed (Text only)              1170  ███████████
  TIER 3 — siRNA by CPC only (Check Abstract)        604  ██████
  TIER 4A — Mixed Tech (siRNA/CPC + Forbidden Term  1369  █████████████
  TIER 4B — Diagnostic/Biomarker only (Review)       145  █
  TIER 5 — Agri/Vet with siRNA CPC (Review)          361  ███
  TIER 6 — No siRNA Signal (Likely Irrelevant)      1872  ██████████████████
  TIER 7 — Agri/Vet without siRNA (Likely Irreleva   296  ██

 There are 1 patent(s) flagged as Needs_Espacenet_Review (unreadable title / missing abstract)

 There are 7 patent(s) flagged with Quantitative Efficacy Data

  Output saved to: EPO_siRNA_IDs_2022_2025_t

,Patent_ID,Priority_Date,Publication_Date,Applicant,Title,Abstract,Espacenet_Link,Tier,Has_Efficacy_Data,Needs_Espacenet_Review,IPCs,CPCs,Family_ID
0,--- TIER 1 — SIRNA CONFIRMED (TEXT + CPC) (158...,,,,,,,,,,,,
1,US2022112494A1,20020925,20220414,UNIV MASSACHUSETTS [US] | UNIVERSITY OF MASSAC...,IN VIVO GENE SILENCING BY CHEMICALLY MODIFIED ...,The present invention provides compositions fo...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A01K2217/075, A61K38/00, A61K48/00, C07D213/69...",32046073
2,US2022315922A1,20021114,20221006,THERMO FISHER SCIENTIFIC INC [US] | Thermo Fis...,Methods and Compositions for Selecting siRNA o...,Efficient sequence specific gene silencing is ...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/713, A61K48/00, C12N15/1048, C12N15/111...",50773796
3,US2022062286A1,20030725,20220303,UNIV SHEFFIELD [GB] | The University of Sheffield,USE OF RNAI INHIBITING PARP ACTIVITY FOR THE M...,The present invention relates to the use of an...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/472, A61K31/517, A61K31/5517, A61K31/70...",27772701
4,US2022119814A1,20040709,20220421,UNIV MASSACHUSETTS [US] | University of Massac...,Therapeutic alteration of transplantable tissu...,"The present invention, at least in part, relat...",https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A01N1/126, A61K47/6911, A61K48/005, C12N15/111...",36125793
5,US2022315945A1,20050916,20221006,MONSANTO TECHNOLOGY LLC [US] | Monsanto Techno...,Methods for genetic control of insect infestat...,The present invention relates to control of pe...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"C07H21/04, C07K14/43536, C07K14/43563, C12N15/...",37497032
6,US2022042021A1,20051229,20220210,ARROWHEAD PHARMACEUTICALS INC [US] | Arrowhead...,RNAi-MEDIATED INHIBITION OF HIF1A FOR TREATMEN...,RNA interference is provided for inhibition of...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/7105, A61K31/713, A61K9/0048, A61P27/00...",38218798
7,US2022112505A1,20070615,20220414,ARROWHEAD PHARMACEUTICALS INC [US] | Arrowhead...,RNAi Inhibition of Alpha-ENaC Expression,The invention relates to compositions and meth...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/713, A61K45/06, A61P11/00, A61P11/06, A...",40130244
8,US2023053332A1,20080902,20230223,ALNYLAM PHARMACEUTICALS INC [US] | LUDWIG INST...,COMPOSITIONS AND METHODS FOR INHIBITING EXPRES...,The invention relates to a double-stranded rib...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/713, A61P35/00, C12N15/1136, C12N15/113...",41268470
9,US2022243199A1,20080925,20220804,ALNYLAM PHARMACEUTICALS INC [US] | Alnylam Pha...,Lipid formulated compositions and methods for ...,The invention relates to a double-stranded rib...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/713, A61P1/00, A61P1/04, A61P1/16, A61P...",41349261


## 4. Full-text XML download

**Reads:** an ID CSV with `Patent_ID` and `Family_ID`.  **Writes:** `eps_xmls/*.xml`, plus `successful_downloads.csv` and `not_in_eps.csv` in the working directory.

This stage downloads full text from the European Publication Server (EPS): description, claims, and the experimental tables that record siRNA activity against target genes. That full text is the input for Section 5.

`xml_download.py` works on the **whole patent family**, not on one publication, because family members are not copies of each other. Divisionals, continuations, and even the A (application) and B (granted) versions of the same application can carry different experimental data. Instead of guessing which relative is best, the module pulls every EP member and leaves the comparison to a later step.

For each patent in the input CSV it:

1. **Fetches the whole family** from EPO OPS, all members and all countries, using both the `/equivalents` service and the `famn=<Family_ID>` family search.
2. **Logs every non-EP member** (US, WO, JP and so on) straight to `not_in_eps.csv`. Only EP publications can have full text on EPS, so these are never requested.
3. **Tests every EP member individually** on EPS, across kind codes (A1, A2, B1 and so on) and publication numbers. Members whose XML has a real full-text structure (`<description>`, `<claims>`, `<table>`) are saved into `eps_xmls/`. Every EP member without full text is written to `not_in_eps.csv` with the reason.

Files already present in `eps_xmls/` are skipped, so an interrupted run can simply be restarted and will fetch only what is missing. A strict 8 second pause follows every EPS and OPS request, which makes a full family sweep deliberately slow.

**Outputs.** `successful_downloads.csv` records what was saved and how each member relates to the requested patent. `not_in_eps.csv` records every member with no EPS full text, which is useful for later coverage analysis.

In [ ]:
from sirna_pipeline.epo.fulltext import download_eps_xmls_with_ops

download_eps_xmls_with_ops(
    csv_filename     = "EPO_siRNA_IDs_2022_2025_only_applicant_Alnylam.csv",
    consumer_key     = CONSUMER_KEY,
    consumer_secret  = CONSUMER_SECRET,
    output_directory = "eps_xmls",
)

Starting FULL-FAMILY extraction for 316 patents...
Enforcing strictly 8+ second delays between all requests.
------------------------------------------------------------

Processing family of: EP4658280A2...
  Family: 3 members (1 EP number(s), 2 non-EP)
  [NO XML] EP4658280 -> no full text on EPS for any kind code (logged).

Processing family of: EP4561631A2...
  Family: 4 members (1 EP number(s), 3 non-EP)
  [NO XML] EP4561631 -> no full text on EPS for any kind code (logged).

Processing family of: EP4594492A1...
  Family: 3 members (1 EP number(s), 2 non-EP)
  [NO XML] EP4594492 -> no full text on EPS for any kind code (logged).

Processing family of: EP4547852A2...
  Family: 4 members (1 EP number(s), 3 non-EP)
  [NO XML] EP4547852 -> no full text on EPS for any kind code (logged).

Processing family of: EP4522742A2...
  Family: 4 members (1 EP number(s), 3 non-EP)
  [NO XML] EP4522742 -> no full text on EPS for any kind code (logged).

Processing family of: EP4547683A2...
  Famil

## 5. Table isolation from the full-text XMLs

**Reads:** `eps_xmls/*.xml`.  **Writes:** `isolated_tables/*.xml`, one file per table.

Each full-text XML is parsed with BeautifulSoup and `lxml` to isolate the experimental tables. `table.py` looks for the `EXAMPLES` heading, the standard EPO boundary between the general description and the experimental section, and then takes every top-level `table` or `tables` element that comes after it and sits inside `<description>`.

For each table it also copies the **five paragraphs immediately before it**, since those usually carry the assay conditions, the cell line and the setup needed to interpret the numbers. Paragraphs from before the `EXAMPLES` heading or outside the description are dropped, and any table nested inside a copied paragraph is removed so the context holds text only. Table plus context are written as one self-contained XML file.

Output filenames follow `<patent_id>_table_<NN>[_T<num>][_in_vitro].xml`:

- `<patent_id>`: base name of the source XML, for example `EP2723758NWB1`.
- `<NN>`: zero-padded position of the table in the document, which keeps ordering stable and filenames unique.
- `T<num>`: the real table number read from the title, for example `T18b`. Omitted when the title has no recognisable `Table <N>` label.
- `_in_vitro`: added when the title mentions any of *antisense strand, cells, in vitro, sense strand, transfection, single dose, dose response, modified sequences, antisense sequence, sense sequence*. This makes the in vitro tables easy to select later.

The cell below runs on a pilot list of ten patents chosen to cover different table layouts, which is enough to test the LLM steps in Sections 6 and 7 at low cost. To process everything instead, uncomment the `glob` line and use it to build `patent_files`.

In [ ]:
from sirna_pipeline.tables.isolate import extract_tables_from_patent
import os, glob

INPUT_DIR = "eps_xmls"          # full-text XMLs downloaded in Section 5
XML_DIR   = "isolated_tables"   # one XML file per extracted table, written here

# EVERY XML downloaded in Section 5.
# patent_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.xml")))

# --- Small test
pilot = [
        "EP2723758NWB1.xml", #1
        "EP2999785NWB1.xml", #2
        "EP3146049NWB1.xml", #3
        "EP3329924NWA1.xml", #4
        "EP3872179NWA1.xml", #5
        "EP3960860NWA2.xml", #6
        "EP4141116NWA1.xml", #7
        "EP2373382NWB1.xml", #8
        "EP4385568NWA2.xml", #9
        "EP4744669NWA2.xml"  #10
]

patent_files = [os.path.join(INPUT_DIR, f) for f in pilot]

for path in patent_files:
    print(f"Processing {os.path.basename(path)}...")
    extract_tables_from_patent(path, output_dir=XML_DIR)

print(f"\nTable isolation complete - {len(patent_files)} patent file(s) processed.")

Processing EP2723758NWB1.xml...
  Saved: isolated_tables\EP2723758NWB1_table_01_T1.xml
  Saved: isolated_tables\EP2723758NWB1_table_02_T2_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_03_T3_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_04_T4_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_05_T5_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_06_T6.xml
  Saved: isolated_tables\EP2723758NWB1_table_07_T7_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_08_T8_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_09_T9_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_10_T10_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_11_T11_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_12_T12_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_13_T13_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_14_T14_in_vitro.xml
  Saved: isolated_tables\EP2723758NWB1_table_15_T15.xml
Processing EP2999785NWB1.xml.

## 6. XML to CSV, with header normalisation

**Reads:** `isolated_tables/`.  **Writes:** `csv_output/<base>_tables.csv` and `csv_output/<base>_context.txt`.

`xml_to_csv.py` turns each isolated table into a structured CSV. It solves two problems at once: pulling data out of CALS-style XML, and turning the heterogeneous, often multi-level headers of patent tables into clean, SQL-compatible column names. Groq keys from the Credentials cell are rotated automatically to stay inside free-tier limits.

Each input XML produces two files:

- `<base>_context.txt`: the table title and the context paragraphs kept in Section 5, plus any full-width annotation rows (method notes, footnotes, spanning captions) that are not column names.
- `<base>_tables.csv`: the table data with normalised headers.

**Structural repairs during extraction.** Three recurring problems are handled. Pseudo-header rows placed in `tbody`, identified by their `namest` and `nameend` spans, are promoted to headers. Empty group-label cells are filled down from the last non-empty value, which restores the cell-line or group labels that `morerows` tables leave blank. Trailing footnote `tgroup`s made of full-width rows are sent to the context file instead of being treated as data.

**Header normalisation, two LLM passes:**

- **Pass 1, structural repair** (`llama-3.3-70b-versatile`): collapses multi-row headers, spanning group labels and `morerows` artefacts into a single flat list. The larger model is used because sparse multi-level layouts, for example `Day 3` and `Transfection (Hep3b)` sitting above `Avg` and `SD`, need reliable merging across rows. This pass is skipped when a table already has one complete header row, and falls back to the rule-based `merge_multilevel_headers` if the API is unavailable.
- **Pass 2, SQL normalisation** (`llama-3.1-8b-instant`): converts the clean header strings into SQL identifiers (lowercase, underscores, `%` becomes `_pct`, `#` becomes `_num`), plus siRNA-specific mappings such as `IC50 (nM)` to `ic50_nm`. Calls are batched per file to save tokens, with the rule-based `basic_sql_normalize` as fallback.

> **Model IDs may need updating.** Groq announced in June 2026 the deprecation of `llama-3.3-70b-versatile` and `llama-3.1-8b-instant` for the free and developer tiers, suggesting `openai/gpt-oss-120b` and `openai/gpt-oss-20b` as replacements. The model IDs are set inside `xml_to_csv.py` (this section) and `core.py` (Section 7), so check the Groq deprecations page before starting a long run.

Three deterministic fixes follow. Duplicate SQL names are made unique by index. A first column of `AD-\d+` duplex identifiers that received a generic name is renamed `duplex_id`. Any header containing `duplex` is normalised to `duplex_id`.

In [ ]:
from sirna_pipeline.tables.parse import convert_directory

convert_directory(
    "isolated_tables",
    output_dir="csv_output",
    api_keys=GROQ_API_KEYS,
)

  [Groq] 4 API key(s) loaded.

Processing: EP2373382NWB1_table_01_T1.xml
  -> EP2373382NWB1_table_01_T1_context.txt  (4 paragraph(s))
  SKIP tables file (no data tables found in EP2373382NWB1_table_01_T1.xml)

Processing: EP2373382NWB1_table_02.xml
  [Pass 2] Normalising 2 unique header(s) to SQL via Groq …
  -> EP2373382NWB1_table_02_context.txt  (4 paragraph(s))
  -> EP2373382NWB1_table_02_tables.csv  (1 table(s))

Processing: EP2373382NWB1_table_03_T2a.xml
  [Pass 2] Normalising 4 unique header(s) to SQL via Groq …
  -> API error: LLM response contained no JSON object.
  [Groq] All attempts exhausted. Using rule-based fallback.
  -> EP2373382NWB1_table_03_T2a_context.txt  (7 paragraph(s))
  -> EP2373382NWB1_table_03_T2a_tables.csv  (1 table(s))

Processing: EP2373382NWB1_table_04_T2b_in_vitro.xml
  [Pass 2] Normalising 4 unique header(s) to SQL via Groq …
  -> EP2373382NWB1_table_04_T2b_in_vitro_context.txt  (7 paragraph(s))
  -> EP2373382NWB1_table_04_T2b_in_vitro_tables.csv  (1 ta

## 7. Primary table assembly

**Reads:** `csv_output/`.  **Writes:** `primary_table*.csv`, `primary_ic50_table*.csv`, `primary_cell_viability_table*.csv`, plus the `failed_tables_*` and `validation_failures_*` manifests.

The last stage consolidates the per-table CSVs into three fixed schemas. For each input CSV, `xml_to_primary_table.py` asks an LLM (`llama-3.3-70b-versatile`) to write a DuckDB `SELECT` that maps that file's particular columns onto the target schema. The query then runs locally in DuckDB, so the model writes the mapping but never touches the data values themselves.

**Routing.** Each CSV is assigned to a schema using keywords from its paired `_context.txt`: `IC50` or `IC 50` goes to the IC50 table, viability keywords go to the cell-viability table, everything else goes to the knockdown table.

| Output file | Content | Key fields |
|---|---|---|
| `primary_table.csv` | Knockdown activity (percent inhibition at a dose) | `duplex_id`, `sense_sequence`, `antisense_sequence`, `cell_line`, `dose_nM`, `inhibition_percent` |
| `primary_ic50_table.csv` | IC50 values, with replicate and timepoint detail | `duplex_id`, `cell_line`, `timepoint_hrs`, `replicate`, `ic50_nM` |
| `primary_cell_viability_table.csv` | Cell-viability screens | `duplex_id`, `cell_line`, `day`, `dose_nM`, `viability_percent` |

**Merging (knockdown table only).** Rows sharing `(patent_id, duplex_id, cell_line, dose_nM)` are merged into one. Annotations (sequences, oligo IDs, target gene) and the measurement fields take the first non-null value, and `source_file` accumulates every contributing filename. Rows carrying only sequences or oligo IDs, with no measurement, are used to enrich matching activity rows and are then dropped, so the output contains no annotation-only records. Both sequence forms are kept: `sense_sequence` and `antisense_sequence` hold the modified form when available, and `*_sequence_unmodified` hold the plain form. The IC50 and viability tables are not merged, since each of their rows is an independent condition.

**Resilience.** The SQL generated for each table is cached on disk, keyed by a SHA hash of the table content and the prompt. If a run stops, for example on a sustained rate limit, restarting reuses the cached SQL for finished tables and calls the API only for the rest.

**Validation.** Every output cell is checked. Numeric fields must be numeric, doses and IC50 values above 10 mM (10⁷ nM) are flagged as implausible, and sequence fields must look like sequences. Failing cells are blanked and listed in a `validation_failures` manifest instead of silently corrupting the output.

In [ ]:
from sirna_pipeline.assembly.build import build_primary_table

build_primary_table(
    "csv_output",
    api_keys=GROQ_API_KEYS,
    per_file_dir="per_file_output",
    file_prefixes=[
        "EP2723758NWB1",
        "EP2999785NWB1",
        "EP3146049NWB1",
        "EP3329924NWA1",
        "EP3872179NWA1",
        "EP3960860NWA2",
        "EP4141116NWA1",
        "EP2373382NWB1",
        "EP4385568NWA2",
        "EP4744669NWA2"
        ]
)

  [Groq] 4 API key(s) loaded.
Found 14 table file(s) across 1 group(s) in 'csv_output'.

[1/1] EP2723758NWB1 — 14 tables

[EP2723758NWB1 table 1/14] Processing: EP2723758NWB1_table_02_T2_in_vitro_tables.csv
  Table type: primary  (measurements=none, via llm)
  Generating SQL...
  Executing SQL via DuckDB...
  → 62 row(s)
  → per-file CSV: per_file_output\EP2723758NWB1_table_02_T2_in_vitro_tables_primary.csv

[EP2723758NWB1 table 2/14] Processing: EP2723758NWB1_table_03_T3_in_vitro_tables.csv
  Table type: primary  (measurements=none, via llm)
  Generating SQL...
  Executing SQL via DuckDB...
  → 62 row(s)
  → per-file CSV: per_file_output\EP2723758NWB1_table_03_T3_in_vitro_tables_primary.csv

[EP2723758NWB1 table 3/14] Processing: EP2723758NWB1_table_04_T4_in_vitro_tables.csv
  Table type: primary  (measurements=['knockdown'], via llm)
  Generating SQL...
  Executing SQL via DuckDB...
  → 128 row(s)
  → per-file CSV: per_file_output\EP2723758NWB1_table_04_T4_in_vitro_tables_primary.csv

## Summary

This notebook implements a reproducible route from the EPO patent corpus to a structured database of siRNA activity data, in seven stages:

1. **Identifier extraction:** two independent search strategies (classification codes, title and abstract keywords) plus an applicant-name query for the Alnylam reference portfolio.
2. **Metadata enrichment:** full bibliographic records through the OPS `/biblio` endpoint, with per-ID retries and an `/abstract` fallback.
3. **Tier filtering:** an eight-tier rule-based classification that guides curation without discarding any record.
4. **Full-text XML download:** the whole EP family for each patent from EPS, with every member lacking full text logged to `not_in_eps.csv`.
5. **Table isolation:** the experimental tables, plus their assay context, lifted out of the EXAMPLES sections.
6. **XML to CSV:** heterogeneous headers normalised into consistent SQL names by a two-pass LLM strategy with rule-based fallbacks.
7. **Primary table assembly:** the per-table CSVs consolidated into knockdown, IC50 and viability tables through LLM-generated DuckDB SQL, with merging, caching and validation.